In [5]:
#==============================
# Check Cuda is Available or Not
#==============================


import torch
print(torch.cuda.is_available())

True


In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/organizations/nih-chest-xrays/data/BBox_List_2017.csv
/kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv


In [1]:
#=============================================
# Dataset.py   //Bridge Between Dataset and NN
#=============================================




import os
import glob
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image


class ChestXrayDataset(Dataset):

    def __init__(
        self,
        csv_file,
        root_dir,
        file_list,
        transform=None
    ):

        self.transform = transform
        self.root_dir = root_dir

        print("Loading CSV...")
        self.data = pd.read_csv(csv_file)

        print("Loading image list...")
        with open(file_list, "r") as f:
            self.image_names = [
                line.strip()
                for line in f.readlines()
            ]

        print("Finding image files...")

        image_paths = glob.glob(
            os.path.join(root_dir, "**", "*.png"),
            recursive=True
        )

        self.image_dict = {
            os.path.basename(path): path
            for path in image_paths
        }

        print(
            f"Total images found: {len(self.image_dict)}"
        )

        self.labels_list = [
            "Atelectasis",
            "Consolidation",
            "Infiltration",
            "Pneumothorax",
            "Edema",
            "Emphysema",
            "Fibrosis",
            "Effusion",
            "Pneumonia",
            "Pleural_Thickening",
            "Cardiomegaly",
            "Nodule",
            "Mass",
            "Hernia"
        ]

        self.label_to_idx = {
            label: idx
            for idx, label
            in enumerate(self.labels_list)
        }

        print("Building label lookup table...")

        self.label_map = {
            row["Image Index"]:
            row["Finding Labels"]
            for _, row
            in self.data.iterrows()
        }

        print("Encoding labels...")

        self.encoded_labels = {}

        for img_name, labels in self.label_map.items():

            label_vec = torch.zeros(
                len(self.labels_list),
                dtype=torch.float32
            )

            for disease in labels.split("|"):

                if disease in self.label_to_idx:

                    label_vec[
                        self.label_to_idx[disease]
                    ] = 1.0

            self.encoded_labels[
                img_name
            ] = label_vec

        print("Dataset ready!")

    def __len__(self):

        return len(
            self.image_names
        )

    def __getitem__(
        self,
        idx
    ):

        img_name = self.image_names[idx]

        img_path = self.image_dict.get(
            img_name
        )

        if img_path is None:

            raise FileNotFoundError(
                f"Image not found: {img_name}"
            )

        try:

            with Image.open(img_path) as img:

                image = img.convert(
                    "RGB"
                )

        except Exception:

            image = Image.new(
                "RGB",
                (224, 224)
            )

        if self.transform:

            image = self.transform(
                image
            )

        label_vec = self.encoded_labels[
            img_name
        ]

        return image, label_vec


if __name__ == "__main__":

    from torchvision import transforms

    DATASET_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    dataset = ChestXrayDataset(
        csv_file=f"{DATASET_ROOT}/Data_Entry_2017.csv",
        root_dir=DATASET_ROOT,
        file_list=f"{DATASET_ROOT}/train_val_list.txt",
        transform=transform
    )

    print(
        "Dataset size:",
        len(dataset)
    )

    image, label = dataset[0]

    print(
        "Image shape:",
        image.shape
    )

    print(
        "Label vector:",
        label
    )

Loading CSV...
Loading image list...
Finding image files...
Total images found: 112120
Building label lookup table...
Encoding labels...
Dataset ready!
Dataset size: 86524
Image shape: torch.Size([3, 224, 224])
Label vector: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])


In [2]:
#=======================================================
# Check train_val_list & test_list file available or not
#=======================================================


import os

ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

print(os.path.exists(f"{ROOT}/train_val_list.txt"))
print(os.path.exists(f"{ROOT}/test_list.txt"))

True
True


In [2]:
#============================
# Train.py  //Training Script
#============================




import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
import seaborn as sns
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# If running as script
# from dataset import ChestXrayDataset

torch.backends.cudnn.benchmark = True


def main():

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print("Using device:", device)

    print("GPU Count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(torch.cuda.get_device_name(i))

# ==========================
# DATASET PATHS
# ==========================

    ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

    CSV = f"{ROOT}/Data_Entry_2017.csv"

    TRAIN_LIST = f"{ROOT}/train_val_list.txt"
    TEST_LIST = f"{ROOT}/test_list.txt"

# ==========================
# TRANSFORMS
# ==========================

    train_transform = transforms.Compose([
        transforms.Resize((224,224)),
        #transforms.RandomCrop((224,224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(5),
        transforms.ColorJitter(
            brightness=0.1,
            contrast=0.1
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])
# ==========================
# DATASET
# ==========================

    train_dataset = ChestXrayDataset(
        CSV,
        ROOT,
        TRAIN_LIST,
        train_transform
    )
    
    test_dataset = ChestXrayDataset(
        CSV,
        ROOT,
        TEST_LIST,
        val_transform
    )


# ==========================
# COMPUTE POSITIVE WEIGHTS
# ==========================
    
    print("Computing class weights...")
    
    all_labels = []
    
    for img_name in train_dataset.image_names:
    
        if img_name in train_dataset.encoded_labels:
    
            all_labels.append(
                train_dataset.encoded_labels[img_name].numpy()
            )

    all_labels = np.array(all_labels)
    
    positive_count = all_labels.sum(axis=0)
    
    negative_count = len(all_labels) - positive_count
    
    pos_weight = torch.tensor(
        negative_count / (positive_count + 1e-6),
        dtype=torch.float32
    )

    pos_weight = torch.clamp(
        pos_weight,
        min=1.0,
        max=100.0
    )

    pos_weight = pos_weight.to(device)
    
    print("Class Weights:")
    print(pos_weight)

    print("\nDisease-wise Class Weights\n")
    
    for disease, weight in zip(
        train_dataset.labels_list,
        pos_weight.cpu().numpy()
    ):
        print(
            f"{disease:<20} {weight:.2f}"
        )

# ==========================
# DATALOADER
# ==========================

    train_loader = DataLoader(
        train_dataset,
        batch_size=192,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=192,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )

# ==========================
# MODEL
# ==========================

    model = models.densenet121(
        weights=models.DenseNet121_Weights.IMAGENET1K_V1
    )
    
    model.classifier = nn.Linear(
        model.classifier.in_features,
        14
    )

    model = model.to(device)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

# ==========================
# LOSS
# ==========================

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=torch.cuda.is_available()
    )

    best_auc = 0
    epochs = 10
    patience = 2
    counter = 0
    
    metrics = []

# ==========================
# TRAINING LOOP
# ==========================

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")

        model.train()

        running_loss = 0

        for images, labels in tqdm(train_loader):

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            with torch.amp.autocast(
                "cuda",
                enabled=torch.cuda.is_available()
            ):

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

            scaler.scale(loss).backward()

            scaler.step(optimizer)

            scaler.update()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        print(
            f"Train Loss: {train_loss:.4f}"
        )


# ==========================
# VALIDATION
# ==========================

        model.eval()

        all_labels = []
        all_outputs = []

        with torch.no_grad():

            for images, labels in test_loader:

                images = images.to(device)

                # Original image prediction
                outputs1 = model(images)
                
                # Horizontally flipped image
                flipped_images = torch.flip(
                    images,
                    dims=[3]
                )
                
                outputs2 = model(flipped_images)
                
                # Average predictions
                probs = (
                    torch.sigmoid(outputs1) +
                    torch.sigmoid(outputs2)
                ) / 2

                all_labels.append(
                    labels.numpy()
                )

                all_outputs.append(
                    probs.cpu().numpy()
                )

        all_labels = np.vstack(all_labels)
        all_outputs = np.vstack(all_outputs)

        try:

            auc = roc_auc_score(
                all_labels,
                all_outputs,
                average="macro"
            )

            print(
                f"Validation AUC: {auc:.4f}"
            )

            metrics.append({
                "epoch": epoch + 1,
                "loss": train_loss,
                "auc": auc
            })

            scheduler.step(auc)

            if auc > best_auc:
            
                best_auc = auc
            
                counter = 0
            
                state_dict = (
                    model.module.state_dict()
                    if isinstance(model, nn.DataParallel)
                    else model.state_dict()
                )

                torch.save({
                    "epoch": epoch + 1,
                    "auc": auc,
                    "model_state_dict": state_dict
                },
                "/kaggle/working/best_model.pth")
            
                print(
                    f"Best model saved! AUC={auc:.4f}"
                )

            else:
            
                counter += 1
            
                print(
                    f"No improvement ({counter}/{patience})"
                )
            
                if counter >= patience:
            
                    print(
                        "Early stopping triggered!"
                    )
            
                    break
        except Exception as e:

            print(
                f"AUC Error: {e}"
            )


    pd.DataFrame(metrics).to_csv(
        "/kaggle/working/training_metrics.csv",
        index=False
    )
    
    print(
        "Metrics CSV saved."
    )
    print(
        f"\nBest AUC: {best_auc:.4f}"
    )


    disease_names = [
        "Atelectasis",
        "Consolidation",
        "Infiltration",
        "Pneumothorax",
        "Edema",
        "Emphysema",
        "Fibrosis",
        "Effusion",
        "Pneumonia",
        "Pleural_Thickening",
        "Cardiomegaly",
        "Nodule",
        "Mass",
        "Hernia"
    ]

    for i, disease in enumerate(disease_names):
    
        try:
    
            disease_auc = roc_auc_score(
                all_labels[:, i],
                all_outputs[:, i]
            )
    
            print(
                f"{disease}: {disease_auc:.4f}"
            )
    
        except Exception as e:
            print(f"{disease}: {e}")


    metrics_df = pd.DataFrame(metrics)

# ==========================
# LOSS CURVE
# ==========================
    
    plt.figure(figsize=(8, 5))
    
    plt.plot(
        metrics_df["epoch"],
        metrics_df["loss"],
        marker="o"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss Curve")
    plt.grid(True)
    
    plt.savefig(
        "/kaggle/working/loss_curve.png",
        bbox_inches="tight"
    )
    
    plt.close()

# ==========================
# AUC CURVE
# ==========================
    
    plt.figure(figsize=(8, 5))
    
    plt.plot(
        metrics_df["epoch"],
        metrics_df["auc"],
        marker="o"
    )

    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.title("Validation AUC Curve")
    plt.grid(True)
    
    plt.savefig(
        "/kaggle/working/auc_curve.png",
        bbox_inches="tight"
    )

    plt.close()
    
    print("Loss curve saved.")
    print("AUC curve saved.")


# ==========================
# ROC CURVE
# ==========================
    
    disease_names = [
        "Atelectasis",
        "Consolidation",
        "Infiltration",
        "Pneumothorax",
        "Edema",
        "Emphysema",
        "Fibrosis",
        "Effusion",
        "Pneumonia",
        "Pleural_Thickening",
        "Cardiomegaly",
        "Nodule",
        "Mass",
        "Hernia"
    ]

    plt.figure(figsize=(10, 8))
    
    for i, disease in enumerate(disease_names):
    
        try:
    
            fpr, tpr, _ = roc_curve(
                all_labels[:, i],
                all_outputs[:, i]
            )
    
            roc_auc = auc(fpr, tpr)
    
            plt.plot(
                fpr,
                tpr,
                label=f"{disease} ({roc_auc:.3f})"
            )
    
        except Exception as e:
            print(f"{disease}: {e}")

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )
    
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves - NIH Chest X-ray")
    plt.legend(fontsize=8)
    plt.grid(True)
    
    plt.savefig(
        "/kaggle/working/roc_curves.png",
        bbox_inches="tight"
    )
    
    plt.close()
    
    print("ROC Curves saved.")



# ==========================
# CLASS DISTRIBUTION PLOT
# ==========================
    
    disease_names = train_dataset.labels_list
    
    plt.figure(figsize=(12,6))
    
    plt.bar(
        disease_names,
        positive_count
    )
    
    plt.xticks(rotation=45)
    
    plt.ylabel("Number of Positive Samples")
    
    plt.title(
        "Class Distribution - NIH Chest X-ray14"
    )

    plt.tight_layout()
    
    plt.savefig(
        "/kaggle/working/class_distribution.png",
        bbox_inches="tight"
    )
    
    plt.close()
    
    print("Class Distribution Saved.")



    # ==========================
    # CONFUSION MATRICES
    # ==========================
    
    threshold = 0.3
    
    for i, disease in enumerate(disease_names):
    
        try:
    
            y_true = all_labels[:, i]
    
            y_pred = (
                all_outputs[:, i] >= threshold
            ).astype(int)
    
            cm = confusion_matrix(
                y_true,
                y_pred
            )

            plt.figure(figsize=(5,4))
    
            sns.heatmap(
                cm,
                annot=True,
                fmt="d",
                cmap="Blues"
            )
    
            plt.title(
                f"{disease} Confusion Matrix"
            )
    
            plt.ylabel("Actual")
    
            plt.xlabel("Predicted")
    
            plt.savefig(
                f"/kaggle/working/{disease}_cm.png",
                bbox_inches="tight"
            )
    
            plt.close()

        except Exception as e:
    
            print(
                f"{disease}: {e}"
            )
    
    print("Confusion matrices saved.")


    print("\nClassification Reports\n")
    for i, disease in enumerate(disease_names):
    
        y_true = all_labels[:, i]
    
        y_pred = (
            all_outputs[:, i] >= threshold
        ).astype(int)
    
        print(f"\n{disease}")
    
        print(
            classification_report(
                y_true,
                y_pred,
                digits=4,
                zero_division=0
            )
        )


#====================
# F1-Score
#====================

    macro_f1 = f1_score(
        all_labels,
        (all_outputs >= threshold).astype(int),
        average="macro",
        zero_division=0
    )
    
    print(f"\nMacro F1 Score: {macro_f1:.4f}")




#====================
# Preciosion & Recall
#====================
    
    macro_precision = precision_score(
        all_labels,
        (all_outputs >= threshold).astype(int),
        average="macro",
        zero_division=0
    )
    
    macro_recall = recall_score(
        all_labels,
        (all_outputs >= threshold).astype(int),
        average="macro",
        zero_division=0
    )
    
    print(
        f"Macro Precision: {macro_precision:.4f}"
    )
    
    print(
        f"Macro Recall: {macro_recall:.4f}"
    )


if __name__ == "__main__":
    main()

Using device: cuda
GPU Count: 2
Tesla T4
Tesla T4
Loading CSV...
Loading image list...
Finding image files...
Total images found: 112120
Building label lookup table...
Encoding labels...
Dataset ready!
Loading CSV...
Loading image list...
Finding image files...
Total images found: 112120
Building label lookup table...
Encoding labels...
Dataset ready!
Computing class weights...
Class Weights:
tensor([  9.4498,  29.3380,   5.2780,  31.8115,  61.7896,  59.8039,  68.1639,
          8.9924,  97.7717,  37.5923,  49.6878,  17.3781,  20.4487, 100.0000],
       device='cuda:0')

Disease-wise Class Weights

Atelectasis          9.45
Consolidation        29.34
Infiltration         5.28
Pneumothorax         31.81
Edema                61.79
Emphysema            59.80
Fibrosis             68.16
Effusion             8.99
Pneumonia            97.77
Pleural_Thickening   37.59
Cardiomegaly         49.69
Nodule               17.38
Mass                 20.45
Hernia               100.00
Downloading: "http

100%|██████████| 30.8M/30.8M [00:00<00:00, 164MB/s] 


Using 2 GPUs

Epoch 1/10


100%|██████████| 451/451 [20:41<00:00,  2.75s/it]

Train Loss: 1.0360


...

Train Loss: 0.8227
Validation AUC: 0.8035
No improvement (1/2)

Epoch 7/10


100%|██████████| 451/451 [20:08<00:00,  2.68s/it]


Train Loss: 0.8027
Validation AUC: 0.8017
No improvement (2/2)
Early stopping triggered!
Metrics CSV saved.

Best AUC: 0.8037
Atelectasis: 0.7462
Consolidation: 0.7406
Infiltration: 0.6998
Pneumothorax: 0.8518
Edema: 0.8327
Emphysema: 0.9095
Fibrosis: 0.8204
Effusion: 0.8190
Pneumonia: 0.7025
Pleural_Thickening: 0.7725
Cardiomegaly: 0.8771
Nodule: 0.7404
Mass: 0.8021
Hernia: 0.9096
Loss curve saved.
AUC curve saved.
Atelectasis: 'numpy.float64' object is not callable
Consolidation: 'numpy.float64' object is not callable
Infiltration: 'numpy.float64' object is not callable
Pneumothorax: 'numpy.float64' object is not callable
Edema: 'numpy.float64' object is not callable
Emphysema: 'numpy.float64' object is not callable
Fibrosis: 'numpy.float64' object is not callable
Effusion: 'numpy.float64' object is not callable
Pneumonia: 'numpy.float64' object is not callable
Pleural_Thickening: 'numpy.float64' object is not callable
Cardiomegaly: 'numpy.float64' object is not callable
Nodule: 'num

/tmp/ipykernel_58/706898480.py:540: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(fontsize=8)


ROC Curves saved.
Class Distribution Saved.
Confusion matrices saved.

Classification Reports


Atelectasis
              precision    recall  f1-score   support

         0.0     0.9690    0.2940    0.4511     22317
         1.0     0.1630    0.9360    0.2777      3279

    accuracy                         0.3762     25596
   macro avg     0.5660    0.6150    0.3644     25596
weighted avg     0.8657    0.3762    0.4289     25596


Consolidation
              precision    recall  f1-score   support

         0.0     0.9909    0.1920    0.3216     23781
         1.0     0.0845    0.9769    0.1555      1815

    accuracy                         0.2476     25596
   macro avg     0.5377    0.5844    0.2386     25596
weighted avg     0.9266    0.2476    0.3098     25596


Infiltration
              precision    recall  f1-score   support

         0.0     0.9210    0.0892    0.1627     19484
         1.0     0.2515    0.9756    0.3999      6112

    accuracy                         0.3009  

In [ ]:
#===================================
# Test Your trained Model
#===================================






import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# ==================================
# PATHS
# ==================================

MODEL_PATH = r"D:\PROJECTS\LUNG-DISEASE-DITECTION\Model\DenseNet121-WB\best_model.pth"

IMAGE_PATH = r"D:\PROJECTS\LUNG-DISEASE-DITECTION\Test Photo\Pneumonia.png"


# ==================================
# DEVICE
# ==================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device:", DEVICE)

# ==================================
# LABELS
# ==================================

LABELS = [
    "Atelectasis",
    "Consolidation",
    "Infiltration",
    "Pneumothorax",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Effusion",
    "Pneumonia",
    "Pleural_Thickening",
    "Cardiomegaly",
    "Nodule",
    "Mass",
    "Hernia"
]

# ==================================
# TRANSFORM
# ==================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# ==================================
# CREATE MODEL
# ==================================



##For ResNet34 Model
# model = models.resnet34(weights=None)

# model.fc = nn.Linear(
#     model.fc.in_features,
#     14
# )


##For DenseNet121 Model
model = models.densenet121(weights=None)

model.classifier = nn.Linear(
    model.classifier.in_features,
    14
)




# ==================================
# LOAD CHECKPOINT
# ==================================

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

print("\nCheckpoint Loaded")
print("Best Epoch :", checkpoint["epoch"])
print("Best AUC   :", checkpoint["auc"])

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(DEVICE)

model.eval()

print("Model Loaded Successfully")

# ==================================
# LOAD IMAGE
# ==================================

image = Image.open(
    IMAGE_PATH
).convert("RGB")

image = transform(image)

image = image.unsqueeze(0)

image = image.to(DEVICE)

# ==================================
# INFERENCE
# ==================================

with torch.no_grad():

    outputs = model(image)

    probabilities = torch.sigmoid(
        outputs
    ).cpu().numpy()[0]

# ==================================
# SORT RESULTS
# ==================================

results = list(
    zip(LABELS, probabilities)
)

results.sort(
    key=lambda x: x[1],
    reverse=True
)

# ==================================
# TOP PREDICTION
# ==================================

best_disease, best_prob = results[0]

print("\n" + "=" * 60)
print("MOST LIKELY DISEASE")
print("=" * 60)

print(
    f"{best_disease} "
    f"({best_prob * 100:.2f}%)"
)

# ==================================
# TOP 5 DISEASES
# ==================================

print("\n" + "=" * 60)
print("TOP 5 POSSIBLE DISEASES")
print("=" * 60)

for disease, prob in results[:5]:

    print(
        f"{disease:<20}"
        f"{prob * 100:.2f}%"
    )

# ==================================
# DISEASE DETECTION
# ==================================

print("\n" + "=" * 60)
print("DETECTED DISEASES (>30%)")
print("=" * 60)

detected = False

for disease, prob in results:

    if prob >= 0.30:

        detected = True

        print(
            f"{disease:<20}"
            f"{prob * 100:.2f}%"
        )

if not detected:

    print(
        "No disease detected with high confidence."
    )

# ==================================
# ALL PROBABILITIES
# ==================================

print("\n" + "=" * 60)
print("ALL DISEASE PROBABILITIES")
print("=" * 60)

for disease, prob in results:

    print(
        f"{disease:<20}"
        f"{prob * 100:.2f}%"
    )